# Xarray-Spatial I/O: Streaming GeoTIFF Writes

When a dask-backed DataArray is passed to `to_geotiff()`, data is written one tile-row at a time.
Only one tile-row lives in memory at once, so you can write rasters larger than RAM without switching to VRT output.

### What you'll build

1. [Generate a chunked dask raster](#data)
2. [Stream-write to a single TIFF](#streaming-tiff)
3. [Stream-write to a VRT mosaic](#streaming-vrt)
4. [Compare compression codecs](#compression)
5. [Understand when streaming does not apply (COG)](#eager-cog)

![Synthetic terrain preview](47_Streaming_GeoTIFF_Write_preview.png)

Import xarray-spatial I/O functions alongside the usual scientific stack.

In [ ]:
%matplotlib inline
import tempfile
import os

import numpy as np
import xarray as xr
import dask.array as da
import matplotlib.pyplot as plt

from xrspatial.geotiff import open_geotiff, to_geotiff

<a id="data"></a>

## Data

A 2000x2000 synthetic terrain surface chunked into 500x500 blocks gives four chunks along each axis (sixteen total).

In [ ]:
rng = np.random.default_rng(1084)
H, W = 2000, 2000

yy, xx = np.meshgrid(
    np.linspace(0, 6 * np.pi, H),
    np.linspace(0, 6 * np.pi, W),
    indexing='ij',
)
terrain = (500 + 200 * np.sin(yy) * np.cos(xx * 0.7)
           + 30 * rng.standard_normal((H, W))).astype(np.float32)

y = np.linspace(45.0, 44.0, H)
x = np.linspace(-122.0, -121.0, W)

raster = xr.DataArray(
    terrain, dims=['y', 'x'],
    coords={'y': y, 'x': x},
    attrs={'crs': 4326, 'nodata': -9999.0},
)

dask_raster = raster.chunk({'y': 500, 'x': 500})
print(f'Shape:  {dask_raster.shape}')
print(f'Chunks: {dask_raster.chunks}')
print(f'dtype:  {dask_raster.dtype}')

The terrain is a sinusoidal surface with added noise, placed in EPSG:4326 coordinates over the Pacific Northwest.
Chunking into 500x500 blocks means each `to_geotiff()` call only needs to hold one tile-row in memory.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
raster.plot.imshow(ax=ax, cmap='terrain', add_colorbar=True)
ax.set_title('Synthetic terrain (2000x2000)')
ax.set_axis_off()
plt.tight_layout()
plt.show()

<a id="streaming-tiff"></a>

## 1. Streaming write to a single TIFF

Pass the dask-backed DataArray to `to_geotiff()` the same way you would a numpy array. The streaming path kicks in automatically.

In [ ]:
tmpdir = tempfile.mkdtemp(prefix='xrs_stream_nb_')

tif_path = os.path.join(tmpdir, 'streamed.tif')
to_geotiff(dask_raster, tif_path)

print(f'File size: {os.path.getsize(tif_path):,} bytes')

# Read back and verify
loaded = open_geotiff(tif_path)
print(f'Shape:  {loaded.shape}')
print(f'CRS:    {loaded.attrs.get("crs")}')
print(f'Match:  {np.allclose(loaded.values, raster.values)}')

Same API, same output, but peak memory was roughly `tile_size * width * 4 bytes` instead of the full 2000x2000 array.

<a id="streaming-vrt"></a>

## 2. Streaming write to a VRT

If you want one tile per dask chunk (useful when chunks are large or you plan to read subregions later), write to a `.vrt` path instead.

In [ ]:
vrt_path = os.path.join(tmpdir, 'tiled.vrt')
to_geotiff(dask_raster, vrt_path)

tiles_dir = os.path.join(tmpdir, 'tiled_tiles')
tile_files = sorted(os.listdir(tiles_dir))
print(f'VRT file:   {os.path.getsize(vrt_path):,} bytes')
print(f'Tile count: {len(tile_files)}')
print(f'Tiles:      {tile_files}')

mosaic = open_geotiff(vrt_path)
print(f'\nMosaic shape: {mosaic.shape}')
print(f'Match:        {np.allclose(mosaic.values, raster.values)}')

Four chunks along each axis produces 16 tile files, stitched by a lightweight XML index.

<a id="compression"></a>

## 3. Compression and layout options

All `to_geotiff` keyword arguments work with the streaming path. Compare codecs to see the file-size difference.

In [ ]:
codecs = ['none', 'deflate', 'zstd', 'lzw']
sizes = {}

for codec in codecs:
    p = os.path.join(tmpdir, f'stream_{codec}.tif')
    to_geotiff(dask_raster, p, compression=codec)
    sizes[codec] = os.path.getsize(p)

for codec, sz in sizes.items():
    ratio = sz / sizes['none']
    print(f'{codec:>8s}: {sz:>12,} bytes  ({ratio:.2%} of uncompressed)')

<a id="eager-cog"></a>

## 4. When streaming doesn't apply

COG output with `cog=True` needs overviews built from the full array. In that case `to_geotiff` falls through to the eager path and calls `.compute()` internally.

In [ ]:
cog_path = os.path.join(tmpdir, 'eager_cog.tif')
to_geotiff(dask_raster, cog_path, cog=True)

print(f'COG size: {os.path.getsize(cog_path):,} bytes')
cog = open_geotiff(cog_path)
print(f'Match:    {np.allclose(cog.values, raster.values)}')

If the full array does not fit in memory, use VRT output instead of COG.

In [ ]:
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)

### Summary

| Write mode | Path | Peak memory | When to use |
|:-----------|:-----|:------------|:------------|
| Streaming TIFF | `out.tif` | ~1 tile-row | Default for dask input |
| Streaming VRT | `out.vrt` | ~1 chunk | Need per-chunk files |
| Eager (COG) | `out.tif`, `cog=True` | Full array | Need overviews |

### References

- [GeoTIFF format specification (OGC)](https://www.ogc.org/standard/geotiff/)
- [Cloud Optimized GeoTIFF (COG) overview](https://www.cogeo.org/)
- [GDAL VRT format documentation](https://gdal.org/en/stable/drivers/raster/vrt.html)
- [Dask array chunking guide](https://docs.dask.org/en/stable/array-chunks.html)
- [xarray-spatial API reference: `to_geotiff`](https://xarray-spatial.readthedocs.io/)